# RVI-Kenya — Notebook 01: Nairobi pilot

This notebook runs the full RVI pipeline end-to-end on the Nairobi river basin
(Nairobi River, Mathare, Ngong, Motoine, Ruiru) and produces:

1. A segment-level RVI map of Nairobi (Folium HTML).
2. A ranked list of the 20 most encroached river segments.
3. A preliminary Flood Hub Spearman correlation scatter plot.

**Exit criterion (proposal §8 Phase 0):** Spearman correlation coefficient
computed with p-value and bootstrap CI.

> **Pre-flight checklist**
> 1. `pip install -e ".[dev]"` from the project root.
> 2. Copy `.env.example` to `.env` and set `FLOODHUB_API_KEY`.
> 3. Have a few hundred MB of disk free for the Microsoft footprint cache.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging, sys
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%H:%M:%S')
sys.path.insert(0, '../src')

from rvi.config import get_config
cfg = get_config()
cfg.ensure_dirs()
print('Pilot bbox:', cfg.nairobi_bbox)
print('Buffer widths (m):', cfg.buffer_widths_m)
print('RVI weights (α, β, γ):', cfg.rvi_alpha, cfg.rvi_beta, cfg.rvi_gamma)
print('Flood Hub key configured:', bool(cfg.floodhub_api_key))

## 1. Stage 1 — Ingestion (waterways)

We fetch OSM waterways within the Nairobi bbox using the Overpass API. The
raw JSON response is cached so re-running the notebook does not hammer Overpass.

In [ ]:
from rvi.ingestion.osm import fetch_waterways_overpass

waterways = fetch_waterways_overpass(
    bbox=cfg.nairobi_bbox,
    config=cfg,
    cache_path=cfg.cache_dir / 'overpass_nairobi.json',
)
print(f'Waterways: {len(waterways)} features')
waterways[['osm_id','waterway','name','name_local','strahler','length_m']].head()

## 2. Stage 2 — Geometry (segmentation)

In [ ]:
from rvi.geometry.segment import segment_waterways

segments = segment_waterways(waterways, config=cfg)
print(f'Segments: {len(segments)}')
print('Mean / median length (m):', segments['segment_length_m'].mean(), segments['segment_length_m'].median())
segments[['segment_id','waterway','strahler','segment_length_m']].head()

## 3. Stage 3 — Ingestion (Microsoft footprints)

This is the slowest step on a fresh machine — a few hundred MB of CSV.gz tile
data is downloaded and cached to `data/cache/`.

In [ ]:
from rvi.ingestion.buildings import load_buildings_for_bbox

buildings = load_buildings_for_bbox(cfg.nairobi_bbox, config=cfg)
print(f'Buildings: {len(buildings)}')
buildings.head()

## 4. Stage 4 — Encroachment + RVI

We compute encroachment for **all three legal buffer widths (6 m, 10 m, 30 m)**
in one pass, then derive the per-segment RVI for each.

In [ ]:
from rvi.analysis.encroachment import detect_encroachment_multi
from rvi.analysis.rvi import compute_rvi_multi

enc_by_width = detect_encroachment_multi(segments, buildings, config=cfg)
rvi_segments = compute_rvi_multi(enc_by_width, config=cfg)
print(rvi_segments.filter(like='rvi_composite').describe())

### Top-20 most encroached segments (30 m buffer)

In [ ]:
top20 = rvi_segments.sort_values('rvi_composite_30m', ascending=False).head(20)
top20[[c for c in ['segment_id','waterway','name','name_local','rvi_composite_30m','n_buildings_30m'] if c in top20.columns]]

## 5. Interactive segment map (Folium)

In [ ]:
from rvi.viz.choropleth import segment_map

fmap = segment_map(rvi_segments, config=cfg, rvi_column='rvi_composite_30m', title='Nairobi pilot — RVI (30 m buffer)')
(cfg.outputs_dir / 'nairobi').mkdir(parents=True, exist_ok=True)
out_html = cfg.outputs_dir / 'nairobi' / 'rvi_nairobi_detail.html'
fmap.save(str(out_html))
print('Saved:', out_html)
fmap

## 6. Stage 5 — Flood Hub validation

We fetch every Kenyan gauge and its current severity, pair each gauge with the
75th-percentile RVI in a 50 km Euclidean radius, and run Spearman ρ with a
non-parametric bootstrap CI.

In [ ]:
from rvi.ingestion.floodhub import FloodHubClient, FloodHubError, gauges_to_geodataframe, statuses_to_dataframe, join_gauges_with_status

try:
    client = FloodHubClient(config=cfg)
    gauges = client.search_gauges_by_area(region_code='KE')
    statuses = client.search_latest_flood_status_by_area(region_code='KE')
    print(f'Gauges: {len(gauges)} — statuses: {len(statuses)}')
except FloodHubError as exc:
    print('Flood Hub unavailable:', exc)
    gauges, statuses = [], []

gauges_geo = gauges_to_geodataframe(gauges)
statuses_df = statuses_to_dataframe(statuses)
gauges_with_status = join_gauges_with_status(gauges_geo, statuses_df) if not gauges_geo.empty else gauges_geo
gauges_with_status.head() if hasattr(gauges_with_status, 'head') else gauges_with_status

In [ ]:
from rvi.analysis.validation import aggregate_upstream_euclidean, correlate_upstream_rvi_to_severity
from rvi.analysis.rvi import compute_rvi

if not gauges_with_status.empty:
    scored30 = compute_rvi(enc_by_width[30.0], config=cfg)
    upstream = aggregate_upstream_euclidean(scored30, gauges_with_status, config=cfg)
    res = correlate_upstream_rvi_to_severity(upstream, config=cfg)
    print(f'Spearman ρ = {res.rho:+.3f}  (n={res.n}, p={res.pvalue:.3g}, 95% CI=[{res.ci_low:+.3f}, {res.ci_high:+.3f}])')
else:
    upstream = None
    res = None
    print('Flood Hub not available — validation skipped.')

In [ ]:
from rvi.viz.choropleth import rvi_severity_scatter

if upstream is not None and res is not None:
    fig = rvi_severity_scatter(upstream, result=res, title='Nairobi pilot — upstream RVI vs Flood Hub severity')
    fig.savefig(cfg.outputs_dir / 'nairobi' / 'rvi_floodhub_correlation.png', dpi=200, bbox_inches='tight')
    fig

## 7. Persist outputs

All artefacts of this run land under `outputs/nairobi/`.

In [ ]:
out_dir = cfg.outputs_dir / 'nairobi'
rvi_segments.to_file(out_dir / 'rvi_segments.gpkg', driver='GPKG')
print('Wrote:', out_dir / 'rvi_segments.gpkg')
if upstream is not None:
    upstream.to_csv(out_dir / 'upstream_30m.csv', index=False)
    print('Wrote:', out_dir / 'upstream_30m.csv')